# DL-DOA Dataset Explorer
**Dataset টা কেমন দেখতে সেটা বোঝার জন্য।**

### Kaggle-এ কীভাবে চালাবে:
1. **'Add Data'** → তোমার dataset attach করো
2. **'Run All'** দাও

### কী দেখাবে:
- Dataset size, shape
- **5টা sample Y matrix** — আসল 16×16 complex values (table) + image
- Ground truth heatmap (256×256)
- Sample-এর metadata (L, SNR, angles)

In [ ]:
# Cell 1 — Library imports
import os, glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

print('✅ Libraries ready')

In [ ]:
# Cell 2 — Dataset খোঁজো (Kaggle input বা working dir)
def find_dataset_files(name_pattern):
    """Search /kaggle/input and /kaggle/working for dataset files."""
    search_roots = ['/kaggle/input', '/kaggle/working', '.', '/content']
    found = []
    for root in search_roots:
        if not os.path.isdir(root): continue
        for fpath in Path(root).rglob(name_pattern):
            found.append(str(fpath))
    return found

# Train dataset খোঁজো
train_data_files = find_dataset_files('train_data.npy') or find_dataset_files('train_data.npz')
train_gt_files   = find_dataset_files('train_gt.npy')   or find_dataset_files('train_gt.npz')
train_meta_files = find_dataset_files('train_meta.npy') or find_dataset_files('train_meta.npz')

print('Train data:', train_data_files)
print('Train GT:  ', train_gt_files)
print('Train meta:', train_meta_files)

if not train_data_files:
    print()
    print('⚠️  Dataset পাওয়া যায়নি!')
    print('   Kaggle → Add Data → তোমার dataset attach করো')
    print('   অথবা নিচের MANUAL_PATH-এ path দাও:')
    MANUAL_PATH = None   # ← এখানে path দাও যদি লাগে, e.g. '/kaggle/input/my-dataset'
else:
    MANUAL_PATH = None

In [ ]:
# Cell 3 — Dataset load করো
def load_arr(path):
    p = Path(path)
    if p.suffix == '.npy':
        return np.load(p, allow_pickle=True)
    else:  # .npz
        npz = np.load(p, allow_pickle=True)
        key = list(npz.files)[0]
        return npz[key]

DATA = load_arr(train_data_files[0])
GT   = load_arr(train_gt_files[0])   if train_gt_files   else None
META = load_arr(train_meta_files[0]) if train_meta_files else None

print(f'Dataset loaded:')
print(f'  DATA  shape: {DATA.shape}   dtype: {DATA.dtype}')
if GT   is not None: print(f'  GT    shape: {GT.shape}')
if META is not None: print(f'  META  shape: {META.shape}  dtype: {META.dtype}')
print(f'\n  Total samples: {len(DATA):,}')
print(f'  Input size:    {DATA.shape[1]}×{DATA.shape[2]}×{DATA.shape[3]}  (H×W×channels)')
if GT is not None:
    print(f'  Output size:   {GT.shape[1]}×{GT.shape[2]}  (heatmap)')

In [ ]:
# Cell 4 — আসল 16×16 Y matrix (5 sample)
# DATA shape: (N, 64, 64, 2)  ← zoom=4 করা আছে P=16 এর জন্য
# Original Y পেতে: data[::4, ::4, :] → 16×16×2

N_SHOW   = 5
STEP     = 4     # zoom factor ছিল 4 (P=16 → 64/16=4)
indices  = np.linspace(0, len(DATA)-1, N_SHOW, dtype=int)

print(f'Showing {N_SHOW} samples (indices: {indices.tolist()})')
print('='*70)

for i, idx in enumerate(indices):
    d = DATA[idx]                                # (64, 64, 2)
    Y_r = d[::STEP, ::STEP, 0]                  # real  part → 16×16
    Y_i = d[::STEP, ::STEP, 1]                  # imag  part → 16×16
    Y   = Y_r + 1j * Y_i                        # complex 16×16

    meta_str = ''
    if META is not None:
        m = META[idx]
        try:
            meta_str = f'  L={m["L"]}  SNR={m["SNR"]}dB  P={m["P"]}'
        except Exception:
            meta_str = f'  meta: {m}'

    print(f'\n── Sample #{idx}{meta_str} ──')
    print(f'   Y matrix (16×16 complex, first 4 rows shown):')

    # Print first 4 rows as table
    for row in range(min(4, Y.shape[0])):
        vals = '  '.join(
            f'{Y[row,col].real:+.2f}{Y[row,col].imag:+.2f}j'
            for col in range(min(6, Y.shape[1]))
        )
        suffix = '  ...' if Y.shape[1] > 6 else ''
        print(f'   row {row:2d}: [{vals}{suffix}]')
    if Y.shape[0] > 4:
        print(f'   ... ({Y.shape[0]-4} more rows)')

    print(f'   |Y| range: [{np.abs(Y).min():.3f}, {np.abs(Y).max():.3f}]')

print('\n✅ Done')

In [ ]:
# Cell 5 — Visual: 5 sample Y matrix (real / imag / magnitude) + GT heatmap

STEP   = 4
N_SHOW = 5
indices = np.linspace(0, len(DATA)-1, N_SHOW, dtype=int)

has_gt = GT is not None
n_cols = 4 if has_gt else 3       # real | imag | |Y| | GT
col_labels = ['Y — Real part', 'Y — Imag part', '|Y| Magnitude']
if has_gt: col_labels.append('Ground Truth Heatmap')

fig, axes = plt.subplots(N_SHOW, n_cols,
                         figsize=(n_cols * 3.5, N_SHOW * 3.2))
fig.suptitle('DL-DOA Dataset — 5 Sample Y Matrices (16×16)', fontsize=14, y=1.01)

for col_idx, lbl in enumerate(col_labels):
    axes[0, col_idx].set_title(lbl, fontsize=10, fontweight='bold', pad=6)

for row_idx, idx in enumerate(indices):
    d  = DATA[idx]
    Yr = d[::STEP, ::STEP, 0]   # 16×16 real
    Yi = d[::STEP, ::STEP, 1]   # 16×16 imag
    Ym = np.sqrt(Yr**2 + Yi**2) # 16×16 magnitude

    meta_str = f'Sample #{idx}'
    if META is not None:
        m = META[idx]
        try:  meta_str += f'  (L={m["L"]}, SNR={m["SNR"]}dB, P={m["P"]})'
        except: pass

    # Real
    ax = axes[row_idx, 0]
    im = ax.imshow(Yr, cmap='RdBu_r', aspect='auto')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_ylabel(meta_str, fontsize=8)
    ax.set_xlabel('Tx beam (P)', fontsize=7)
    ax.set_ylabel('Rx beam (Q)', fontsize=7)
    ax.text(0.02, 0.98, meta_str, transform=ax.transAxes,
            fontsize=7, va='top', color='black',
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.7))

    # Imag
    ax = axes[row_idx, 1]
    im = ax.imshow(Yi, cmap='RdBu_r', aspect='auto')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xlabel('Tx beam (P)', fontsize=7)
    ax.set_ylabel('Rx beam (Q)', fontsize=7)

    # Magnitude
    ax = axes[row_idx, 2]
    im = ax.imshow(Ym, cmap='hot', aspect='auto')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_xlabel('Tx beam (P)', fontsize=7)
    ax.set_ylabel('Rx beam (Q)', fontsize=7)

    # Ground Truth heatmap
    if has_gt:
        ax = axes[row_idx, 3]
        gt = GT[idx, :, :, 0] if GT[idx].ndim == 3 else GT[idx]
        im = ax.imshow(gt, cmap='viridis', aspect='auto', origin='lower')
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ax.set_xlabel('ωφ (AoD frequency)', fontsize=7)
        ax.set_ylabel('ωψ (AoA frequency)', fontsize=7)

plt.tight_layout()
plt.savefig('/kaggle/working/dataset_explorer.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: /kaggle/working/dataset_explorer.png')

In [ ]:
# Cell 6 — Full 16×16 Y matrix print করো (1টা sample, সব values)
SHOW_IDX = 0    # ← যেটা দেখতে চাও সেই index দাও
STEP = 4

d  = DATA[SHOW_IDX]
Yr = d[::STEP, ::STEP, 0]   # 16×16
Yi = d[::STEP, ::STEP, 1]   # 16×16
Y  = Yr + 1j * Yi

print(f'Sample #{SHOW_IDX} — Full 16×16 Y matrix (complex values):')
print(f'Shape: {Y.shape}   |Y| range: [{np.abs(Y).min():.4f}, {np.abs(Y).max():.4f}]')
print()

# Header
header = 'row\\col  ' + '  '.join(f'  p={j:2d}  ' for j in range(Y.shape[1]))
print('      ' + '  '.join(f'    p={j:2d}   ' for j in range(Y.shape[1])))
print('-' * (12 * Y.shape[1]))
for r in range(Y.shape[0]):
    row_str = f'q={r:2d}  |  '
    for c in range(Y.shape[1]):
        val = Y[r, c]
        row_str += f'{val.real:+6.3f}{val.imag:+6.3f}j  '
    print(row_str)

print()
print('q = Rx beam index (0..15), p = Tx beam index (0..15)')

In [ ]:
# Cell 7 — Dataset statistics
print('='*50)
print('Dataset Statistics')
print('='*50)
print(f'Total samples:     {len(DATA):,}')
print(f'Input shape:       {DATA.shape[1:]}  (upsampled 64×64×2)')
print(f'Original Y shape:  16×16×2           (before zoom)')

# Value range
print(f'\nInput value stats:')
print(f'  Real part — min: {DATA[:,:,:,0].min():.4f}  max: {DATA[:,:,:,0].max():.4f}  mean: {DATA[:,:,:,0].mean():.4f}')
print(f'  Imag part — min: {DATA[:,:,:,1].min():.4f}  max: {DATA[:,:,:,1].max():.4f}  mean: {DATA[:,:,:,1].mean():.4f}')

if GT is not None:
    print(f'\nGround truth shape: {GT.shape[1:]}  (256×256 heatmap)')
    print(f'GT value stats:   min={GT.min():.6f}  max={GT.max():.4f}')

if META is not None:
    print(f'\nMetadata (first 5 samples):')
    for i in range(min(5, len(META))):
        try:
            m = META[i]
            print(f'  [{i}] L={m["L"]}  SNR={m["SNR"]:3d}dB  P={m["P"]}  phi={[f"{a:.2f}" for a in m["phi"]]}  psi={[f"{a:.2f}" for a in m["psi"]]}')
        except Exception as e:
            print(f'  [{i}] {META[i]}  ({e})')

print('\n✅ Exploration complete!')